[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/10_ranking_report/10_ranking_lab.ipynb)


# 10 — 후보 랭킹 · 리포트 — 앞 랩 산출물을 하나의 순위로

> 본문 [`10_ranking_report.md`](10_ranking_report.md) 와 **한 절씩 짝지어** 보세요.
> **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 진행해요.
> 이 노트북의 표·그래프·수치는 **여러분이 방금 돌린 `my_run/`** 에서 나와요. 실행을 건너뛰거나 실패하면 저장소에 커밋된 `data/`(실제 실행 산출물) 로 자동 폴백해요.
>
> **실측 소요 —** Sapiens 재스코어링(후보 5종) **2초 미만** · ANARCI germline(10 체인) **0.42초**

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

- **Colab**: 이 셀을 그대로 실행하세요. 저장소를 클론하고 이 챕터(`10_ranking_report`) 폴더로 이동한 뒤, 필요한 패키지만 깝니다.
- **로컬**: 이 노트북을 `10_ranking_report/` 폴더 안에서 열었다면 클론 없이 그대로 진행돼요.

이 셀이 끝나면 두 개의 경로가 준비돼요 — **`my_run/`**(내가 직접 만들 결과)과 **`data/`**(저장소에 커밋된 레퍼런스 결과).
아래 랩은 항상 `my_run/` 을 먼저 찾고, 없으면 `data/` 로 폴백하면서 **어느 쪽을 쓰는지 print** 합니다.
- ANARCI 는 numbering 을 **hmmscan(HMMER)** 서브프로세스로 돌려요. Colab 에서는 `apt-get install -y hmmer` 가 함께 실행돼요 — pip 만으로는 `hmmscan` 이 없어 죽어요.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "10_ranking_report"
APT_PKGS = "hmmer"     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas matplotlib pyyaml sapiens anarci abnumber"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막아요.
# (멈춤은 예외가 안 나서 폴백이 안 걸려요 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어져요)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았어요. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

# ANARCI 는 numbering 을 hmmscan(HMMER) 서브프로세스로 돌려요 — pip 만으로는 hmmscan 이 없어요.
if APT_PKGS and IN_COLAB:
    _run("apt-get -qq update")                 # 인덱스가 낡으면 install 이 404 로 죽는다
    _run(f"apt-get -qq install -y {APT_PKGS}")
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨져요.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했어요."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — 앞 랩의 my_run 산출물 모으기

Ch.05~09 에서 **내가 직접 만든** 후보·지표를 끌어와요. 빠진 것만 `data/` 레퍼런스로 채워요(어느 쪽인지 print).


In [ ]:
import pandas as pd, numpy as np

SOURCES = {}

cands = {"parental": (VH, VL)}
for chapter, fname, keys, label in [
    ("05_humanize_sapiens", "sapiens_humanized_noguard.fasta", ("sapiens_humanized_VH", "sapiens_humanized_VL"), "sapiens"),
    ("06_cdr_safe_tools",   "humatch_humanised.fasta",         ("humatch_humanised_VH", "humatch_humanised_VL"), "humatch"),
    ("06_cdr_safe_tools",   "anthroab_best_score.fasta",       ("anthroab_bestscore_VH", "anthroab_bestscore_VL"), "anthroab"),
    ("06_cdr_safe_tools",   "anthroab_masked_FRonly.fasta",    ("anthroab_masked_VH", "anthroab_masked_VL"), "anthroabFRmasked"),
]:
    p = GUIDE_ROOT / chapter / "my_run" / fname
    if p.exists():
        f = read_fasta(p); cands[label] = (f[keys[0]], f[keys[1]]); SOURCES[label] = f"내 결과 · {chapter}"

if len(cands) == 1:
    v = read_fasta("data/variants.fasta")
    for n in ["sapiens", "humatch", "anthroab", "anthroabFRmasked"]:
        cands[n] = (v[f"{n}_VH"], v[f"{n}_VL"]); SOURCES[n] = "레퍼런스 · data/variants.fasta"

for k, v in SOURCES.items():
    print(f"  {k:18s} ← {v}")
print("\n후보:", ", ".join(cands))

## 2) 직접 실행 — 지표 6종 계산

| 항목 | 어디서 |
|---|---|
| humanness | Sapiens 재스코어링(정의 b) — 이 셀에서 **직접 계산**(후보 5종 2초 미만) |
| nativeness | AbNatiV1 (Ch.07 my_run → 없으면 data/) |
| naturalness | Ab-RoBERTa paired (Ch.07 my_run → 없으면 data/) |
| germline identity | ANARCI `--assign_germline` — 이 셀에서 **직접 실행**(10 체인 0.42초) |
| CDR 보존 | parental CDR 문자열이 후보에 그대로 있는가(indel 안전) |
| developability | liability 신규 모티프(Ch.09 방식으로 직접 계산) |
| 구조 | CDR-H3 RMSD (Ch.08 my_run → 없으면 data/; **Sapiens 후보만** 측정돼 있음) |

In [ ]:
import re, subprocess, tempfile

# (1) humanness — 직접 재스코어링 (정의 b)
hum = {}
try:
    import sapiens

    def mean_self_prob(seq, chain):
        m = sapiens.predict_scores(seq, chain)
        return float(np.mean([m.loc[i, aa] for i, aa in enumerate(seq)]))

    t0 = time.time()
    for n, (h, l) in cands.items():
        ph, pl = mean_self_prob(h, "H"), mean_self_prob(l, "L")
        hum[n] = (ph * len(h) + pl * len(l)) / (len(h) + len(l))   # 길이가중 paired
    print(f"humanness(Sapiens 재스코어링) {len(cands)}후보: {time.time()-t0:.1f}초")
except Exception as e:
    print("Sapiens 재스코어링 실패:", type(e).__name__, str(e)[:120])
    print("[레퍼런스] data/humanness_all_candidates.csv")
    ref = pd.read_csv("data/humanness_all_candidates.csv")
    hum = ref[ref.chain == "paired"].set_index("candidate")["mean_self_prob"].to_dict()
    hum = {n: hum.get(n, np.nan) for n in cands}

# (2) germline identity — ANARCI 직접 실행
t0 = time.time()
germ = {}
with tempfile.TemporaryDirectory() as td:
    td = pathlib.Path(td)
    write_fasta(td / "all.fa", {f"{n}_{c}": s for n, (h, l) in cands.items()
                                for c, s in (("VH", h), ("VL", l))})
    try:
        r = subprocess.run(["ANARCI", "-i", str(td / "all.fa"), "-s", "imgt", "--csv",
                            "-o", str(td / "gl"), "--assign_germline", "--use_species", "human"],
                           capture_output=True, text=True)
        rc = r.returncode
    except FileNotFoundError:
        rc = 127          # ANARCI/hmmscan 이 PATH 에 없음 → 레퍼런스로 폴백
    if rc == 0:
        vs = {}
        for f in sorted(td.glob("gl_*.csv")):
            for _, row in pd.read_csv(f).iterrows():
                n = row["Id"].rsplit("_", 1)[0]
                vs.setdefault(n, []).append(float(row["v_identity"]))
        germ = {n: float(np.mean(v)) for n, v in vs.items()}
        print(f"germline identity(ANARCI) {len(cands)*2} 체인: {time.time()-t0:.2f}초")
    else:
        ref = pd.read_csv("data/germline_all_candidates.csv")
        germ = ref.groupby("candidate")["v_identity"].mean().to_dict()
        print("[레퍼런스] data/germline_all_candidates.csv")

# (3) nativeness (AbNatiV1 VH) · (4) naturalness (Ab-RoBERTa paired)
abn = pd.read_csv(find_prev("07_nativeness", "abnativ_summary_all_models.csv", quiet=True))
abn1 = abn[(abn.model_generation == "AbNatiV1") & (abn.variant.str.endswith("_VH"))]
nat = {r.variant.split("_")[0]: float(r.overall_score) for r in abn1.itertuples()}

abr = pd.read_csv(find_prev("07_nativeness", "abroberta_scores_summary.csv", quiet=True))
ntr = abr[abr.chain == "paired"].set_index("variant")["mean_logp"].to_dict()

# (5) CDR 보존 · (6) liability 신규
ct = pd.read_csv("data/cdr_table_imgt.csv")
cdrs = [(r["chain"], r["sequence"]) for _, r in ct.iterrows()]
MOTIFS = {"N-glycosylation": r"N[^P][ST]", "deamidation": r"N[GS]",
          "isomerization": r"DG", "oxidation": r"[MW]"}
def scan(seq):
    return {m: {x.start() + 1 for x in re.finditer(p, seq)} for m, p in MOTIFS.items()}
par_scan = {"VH": scan(VH), "VL": scan(VL)}

met = []
for n, (h, l) in cands.items():
    kept = sum(1 for chain, s in cdrs if s in (h if chain == "H" else l))
    new_g = len(scan(h)["N-glycosylation"] - par_scan["VH"]["N-glycosylation"]) + \
            len(scan(l)["N-glycosylation"] - par_scan["VL"]["N-glycosylation"])
    new_all = sum(len(scan(h)[m] - par_scan["VH"][m]) + len(scan(l)[m] - par_scan["VL"][m])
                  for m in MOTIFS)
    met.append({"candidate": n, "humanness": round(hum[n], 4),
                "nativeness_AbNatiV1_VH": round(nat.get(n, np.nan), 4) if n in nat else np.nan,
                "naturalness_AbRoBERTa": round(ntr.get(n, np.nan), 4) if n in ntr else np.nan,
                "germline_V_identity": round(germ.get(n, np.nan), 4) if n in germ else np.nan,
                "CDR_kept": kept, "new_glyc": new_g, "new_liabilities": new_all})

# (7) 구조 — Sapiens 후보만 측정돼 있음
rm = pd.read_csv(find_prev("08_structure", "cdr_h3_rmsd_summary.csv", quiet=True))
h3 = float(rm[rm.metric.str.startswith("cdr_h3")]["value_angstrom"].iloc[0])
mt = pd.DataFrame(met)
mt["cdr_h3_rmsd"] = [0.0 if n == "parental" else (h3 if n == "sapiens" else np.nan)
                     for n in mt["candidate"]]
mt.to_csv(MY / "metrics_table.csv", index=False)
display(mt)
print("cdr_h3_rmsd 는 Ch.08 에서 접어 본 후보(parental·Sapiens)만 값이 있어요 — 나머지는 NaN(가중합에서 제외).")

## 3) 직접 실행 — hard filter + 가중합 랭킹 (본문 10.1)

본문 10.1.1 의 가중치를 **실제로 측정한 항목**에 맞춰 재배분했어요(측정하지 않은 항목에 점수를 주지 않기 위해).

```
Final score = 0.30 humanness + 0.25 nativeness + 0.20 germline identity
            + 0.15 developability(신규 liability 적을수록) + 0.10 naturalness
```

**Hard filter (즉시 탈락)** — ① CDR 6개 중 하나라도 파손 ② 신규 N-glycosylation 모티프 ③ humanness 가 parental 이하.

In [ ]:
def minmax(s):
    s = s.astype(float)
    if s.notna().sum() == 0 or s.max() == s.min():
        return pd.Series([0.5] * len(s), index=s.index)
    return (s - s.min()) / (s.max() - s.min())

W = {"humanness": 0.30, "nativeness_AbNatiV1_VH": 0.25, "germline_V_identity": 0.20,
     "developability": 0.15, "naturalness_AbRoBERTa": 0.10}

rank = mt.set_index("candidate").copy()
rank["developability"] = -rank["new_liabilities"]            # 적을수록 좋음 → 부호 반전
missing = {k: list(rank.index[rank[k].isna()]) for k in W if rank[k].isna().any()}
if missing:
    print("측정값이 없는 항목(0점 처리):", missing)
norm = pd.DataFrame({k: minmax(rank[k]).fillna(0.0) for k in W})
rank["score"] = sum(norm[k] * w for k, w in W.items()).round(4)

par_hum = float(rank.loc["parental", "humanness"])
rank["hard_filter"] = [
    "; ".join(filter(None, [
        "CDR 파손" if r.CDR_kept < 6 else "",
        "신규 N-glyc" if r.new_glyc > 0 else "",
        "humanness 미개선" if (r.humanness <= par_hum and i != "parental") else "",
    ])) or "pass"
    for i, r in rank.iterrows()]
rank.loc["parental", "hard_filter"] = "(baseline)"

out = rank.sort_values("score", ascending=False)[
    ["score", "hard_filter", "humanness", "nativeness_AbNatiV1_VH", "germline_V_identity",
     "naturalness_AbRoBERTa", "CDR_kept", "new_glyc", "new_liabilities", "cdr_h3_rmsd"]]
out.to_csv(MY / "ranking.csv")
display(out)

adv = out[(out.hard_filter == "pass")]
print("\nhard filter 통과:", ", ".join(adv.index) if len(adv) else "없음")
print("주의 — 점수 1위여도 hard filter 에 걸리면 실험으로 넘기지 않아요(또는 CDR backmutation 후 재평가).")

## 4) 직접 실행 — candidate report (CSV + GuideDB YAML, 본문 10.1.3 · 10.2)

In [ ]:
import yaml

top = adv.index[0] if len(adv) else out.index[0]
h, l = cands[top]

def mutations(par, cand):
    if len(par) != len(cand):
        return ["(길이 다름 — indel 포함, IMGT 정렬 필요)"]
    return [f"{a}{i+1}{b}" for i, (a, b) in enumerate(zip(par, cand)) if a != b]

report_rows = []
for n in out.index:
    if n == "parental":
        continue
    ch, cl = cands[n]
    r = out.loc[n]
    report_rows.append({
        "Candidate ID": f"HZ_{n}_01", "Method": n,
        "VH mutations": ", ".join(mutations(VH, ch))[:120],
        "VL mutations": ", ".join(mutations(VL, cl))[:120],
        "CDR mutations": "none (보호)" if r.CDR_kept == 6 else f"{6 - int(r.CDR_kept)}개 CDR 파손",
        "Humanness": r.humanness, "AbNatiV1 VH": r["nativeness_AbNatiV1_VH"],
        "Germline V identity": r["germline_V_identity"],
        "Developability": "clean" if r.new_liabilities == 0 else f"신규 liability {int(r.new_liabilities)}",
        "CDR-H3 RMSD": r.cdr_h3_rmsd,
        "Final score": r.score,
        "Recommendation": "advance" if r.hard_filter == "pass" else f"reject/backmutate ({r.hard_filter})",
    })
rep = pd.DataFrame(report_rows)
rep.to_csv(MY / "candidate_report.csv", index=False)
display(rep[["Candidate ID", "Method", "CDR mutations", "Humanness", "Final score", "Recommendation"]])

gl_ref = pd.read_csv("data/germline_assignment.csv")
db = {
    "project": {"id": "HZ_running_example", "parent_clone": "parental", "date": time.strftime("%Y-%m-%d")},
    "input_sequences": {"heavy": {"name": "parental_H", "sequence": VH},
                        "light": {"name": "parental_L", "sequence": VL}},
    "annotation": {
        "numbering_scheme": "IMGT", "cdr_definition": "IMGT",
        "heavy_germline": str(gl_ref[(gl_ref.chain == "H") & (gl_ref.gene_type == "V")]["gene"].iloc[0]),
        "light_germline": str(gl_ref[(gl_ref.chain == "L") & (gl_ref.gene_type == "V")]["gene"].iloc[0]),
        "heavy_cdr3": str(ct[(ct.chain == "H") & (ct.cdr == "CDR3")]["sequence"].iloc[0]),
        "note": "J-gene 은 IGHJ6*01/IGHJ4*01 동점(85.71%) — 도구 tie-break 차이",
    },
    "candidates": [
        {"id": f"HZ_{n}_01", "method": n,
         "sequences": {"heavy": cands[n][0], "light": cands[n][1]},
         "scores": {"humanness_sapiens_rescored": float(out.loc[n, "humanness"]),
                    "nativeness_abnativ1_vh": (None if pd.isna(out.loc[n, "nativeness_AbNatiV1_VH"])
                                               else float(out.loc[n, "nativeness_AbNatiV1_VH"])),
                    "naturalness_abroberta_paired": (None if pd.isna(out.loc[n, "naturalness_AbRoBERTa"])
                                                     else float(out.loc[n, "naturalness_AbRoBERTa"])),
                    "germline_v_identity": float(out.loc[n, "germline_V_identity"]),
                    "final_score": float(out.loc[n, "score"])},
         "structure": {"cdr_h3_rmsd": (None if pd.isna(out.loc[n, "cdr_h3_rmsd"])
                                       else float(out.loc[n, "cdr_h3_rmsd"]))},
         "cdr_preserved": int(out.loc[n, "CDR_kept"]) == 6,
         "decision": "advance" if out.loc[n, "hard_filter"] == "pass" else "reject/backmutate"}
        for n in out.index if n != "parental"
    ],
}
(MY / "candidate_report.yaml").write_text(yaml.safe_dump(db, allow_unicode=True, sort_keys=False))
print("\n→", MY / "candidate_report.csv")
print("→", MY / "candidate_report.yaml")
print((MY / "candidate_report.yaml").read_text()[:700], "...")

## 5) 레퍼런스 대조 — 같은 코드, 레퍼런스 입력

내 후보 대신 **커밋된 레퍼런스 후보(`data/`)** 로 같은 지표를 읽어 랭킹이 어떻게 나오는지 비교해요.
지표 자체가 실측이므로, 실행을 다 건너뛴 사람도 여기서 같은 결론에 도달해요.

In [ ]:
ref_hum  = pd.read_csv("data/humanness_all_candidates.csv")
ref_hum  = ref_hum[ref_hum.chain == "paired"].set_index("candidate")["mean_self_prob"]
ref_germ = pd.read_csv("data/germline_all_candidates.csv").groupby("candidate")["v_identity"].mean()
ref_abn  = pd.read_csv("data/abnativ_summary_all_models.csv")
ref_abn  = ref_abn[(ref_abn.model_generation == "AbNatiV1") & (ref_abn.variant.str.endswith("_VH"))]
ref_abn  = {r.variant.split("_")[0]: r.overall_score for r in ref_abn.itertuples()}
ref_abr  = pd.read_csv("data/abroberta_scores_summary.csv")
ref_abr  = ref_abr[ref_abr.chain == "paired"].set_index("variant")["mean_logp"]

ref_tbl = pd.DataFrame({"humanness": ref_hum.round(4), "nativeness_AbNatiV1_VH": pd.Series(ref_abn).round(4),
                        "germline_V_identity": ref_germ.round(4),
                        "naturalness_AbRoBERTa": ref_abr.round(4)})
print("[레퍼런스 지표 — 전부 실제 실행 산출물]")
display(ref_tbl)

print("\n내 랭킹 순서            :", " > ".join(out.index))
print("레퍼런스 humanness 순서 :", " > ".join(ref_hum.sort_values(ascending=False).index))
print("레퍼런스 nativeness 순서:", " > ".join(pd.Series(ref_abn).sort_values(ascending=False).index))
print("\n두 순서가 다르면? 정상이에요 — humanness 와 nativeness 는 다른 축이고, "
      "랭킹은 거기에 germline·developability·CDR 보존을 더해 계산하니까요.")

## 이 랩에서 확인한 것

1. 랭킹은 **여러 축의 가중합 + hard filter** 입니다. 점수 1위여도 CDR 이 파손됐거나 신규 N-glyc 모티프가 생겼으면 **탈락**입니다.
2. 실측 지표(후보 5종).
   - **humanness**(Sapiens 재스코어링, paired) — parental 0.7303 · Sapiens **0.8424** · Humatch 0.7988 · AnthroAb 0.7941 · AnthroAb(FR-masked) 0.7136
   - **nativeness**(AbNatiV1 VH) — parental 0.6477 · Sapiens **0.8803** · Humatch 0.8305 · AnthroAb 0.8064
   - **germline V identity** — parental VH 0.63 → Sapiens/Humatch **0.80**
   - **CDR 보존** — Humatch 6/6, AnthroAb(FR-masked) 6/6, Sapiens·AnthroAb(best_score) 는 CDR-L1 파손
3. 그래서 "가장 사람다운 후보"와 "실험으로 넘길 후보"가 **다를 수 있어요.** Sapiens 후보를 쓰려면 Ch.05 의 **CDR 가드 적용본**을 써야 합니다.
4. 산출물 — `my_run/metrics_table.csv` · `ranking.csv` · `candidate_report.csv` · `candidate_report.yaml`(GuideDB 스키마).

전체 체크리스트·용어집 → **[Ch.11 부록](../11_appendix/11_appendix.md)**